# Hybrid indexing (chunker + embedder + OpenSearch) — service test

Tests `TextChunker` in isolation, then `HybridIndexingService.index_paper` end-to-end.

**Prereqs:** `docker compose up -d postgres opensearch` + at least one paper with raw_text in Postgres. 
**Cost:** embeds ~20 chunks (~$0.0005).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# GOTCHA: Paper import must precede make_database
from src.models.paper import Paper
from src.db.factory import make_database

db = make_database()
with db.get_session() as session:
    paper = session.query(Paper).filter(Paper.raw_text.isnot(None)).first()
    assert paper, 'No parsed papers in DB — run the DAG or e2e notebook first'
    paper_data = {
        'id': str(paper.id), 'arxiv_id': paper.arxiv_id, 'title': paper.title,
        'authors': paper.authors, 'abstract': paper.abstract,
        'categories': paper.categories, 'published_date': paper.published_date,
        'raw_text': paper.raw_text, 'sections': paper.sections,
    }
print(paper.arxiv_id, '|', len(paper.raw_text), 'chars of raw_text')

In [ ]:
# Step 1 in isolation: chunking only (no embedding, free)
from src.services.indexing.text_chunker import TextChunker

chunker = TextChunker()
chunks = chunker.chunk_paper(
    title=paper_data['title'], abstract=paper_data['abstract'],
    full_text=paper_data['raw_text'], arxiv_id=paper_data['arxiv_id'],
    paper_id=paper_data['id'], sections=paper_data['sections'],
)
print(len(chunks), 'chunks')
for c in chunks[:3]:
    print(f"  [{c.metadata.chunk_index}] {c.metadata.section_title[:30]:30} | {c.metadata.word_count} words")

In [ ]:
# Steps 1-3 together: chunk → embed → bulk-index into OpenSearch
from src.services.indexing.factory import make_hybrid_indexing_service

indexer = make_hybrid_indexing_service()
stats = await indexer.index_paper(paper_data)
stats   # chunks_indexed should equal chunks_created, errors 0

In [ ]:
# Verify: count this paper's chunks in OpenSearch
from src.services.opensearch.factory import make_opensearch_client

os_client = make_opensearch_client()
n = os_client.client.count(
    index=os_client.index_name,
    body={'query': {'term': {'arxiv_id': paper_data['arxiv_id']}}},
)['count']
print(f"chunks in OpenSearch for {paper_data['arxiv_id']}: {n}")